## Statistical Test: Does the Education Program Improve Learning?

To determine whether the education program is associated with a meaningful difference in learning scores, we can compare the average learning scores of two groups:

- **Program group:** Guests who participated in the education program.
- **Control group:** Guests who did not participate in the education program.

A **two-sample t-test** is used to test whether the difference between the two group means is statistically significant.

### Hypotheses

**Null hypothesis (H₀):**  
There is no difference in the average learning score between guests who participated in the education program and those who did not.

**Alternative hypothesis (H₁):**  
There is a difference in the average learning score between the two groups.

We use a significance level of **0.05**. If the p-value is less than 0.05, we reject the null hypothesis and consider the difference statistically significant.

The Welch's t-test (`equal_var=False`) is used because it does not assume that the two groups have equal variance.

In [7]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import ttest_ind
import statsmodels.formula.api as smf

In [2]:
data = pd.read_csv("../data/guest_survey_data.csv")
data

,guest_id,age,family_size,first_visit,visit_frequency,visit_motivation,venue,education_program,spending,satisfaction,learning_score,likelihood_return,likelihood_recommend,participated_booth
0,10001,56,3,no,First visit,Entertainment,Garden,No,14.617588,5.0,1.0,5.0,5.0,5
1,10002,69,5,no,Quarterly,Entertainment,Curiosity Museum,Yes,11.742618,5.0,3.0,5.0,5.0,5
2,10003,46,4,yes,Quarterly,Exhibits,Farm,No,15.875554,5.0,3.0,5.0,5.0,3
3,10004,32,5,no,Quarterly,Education,Farm,Yes,5.942116,4.0,2.0,3.0,4.0,3
4,10005,60,5,no,First visit,Family Time,Garden,No,9.851911,4.0,2.0,4.0,4.0,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1195,11196,23,6,no,First visit,Nature,Dinosaurs,No,6.918145,5.0,2.0,5.0,4.0,5
1196,11197,43,5,no,First visit,Family Time,Curiosity Museum,No,4.196820,5.0,3.0,4.0,5.0,1
1197,11198,71,6,no,First visit,Family Time,Dinosaurs,Yes,0.000000,5.0,3.0,5.0,4.0,5
1198,11199,21,3,no,First visit,Education,Garden,No,14.380630,4.0,3.0,5.0,4.0,2


In [3]:
program_group = data[data["education_program"] == "Yes"]["learning_score"]
control_group = data[data["education_program"] == "No"]["learning_score"]

t_stat, p_value = ttest_ind(program_group,control_group,equal_var=False)

print("t-statistic:", t_stat)
print("p-value:", p_value)

if p_value < 0.05:
    print("The difference is statistically significant.")
else:
    print("The difference is not statistically significant.")

t-statistic: 12.71521444367223
p-value: 1.4615568234103225e-34
The difference is statistically significant.


### Interpretation

The **p-value** tells us how strong the evidence is against the null hypothesis.

- If **p < 0.05**, the difference in learning scores is statistically significant. This suggests that the difference between the program and control groups is unlikely to be explained by random variation alone.
- If **p ≥ 0.05**, the difference is not statistically significant. We do not have enough evidence to conclude that the education program is associated with a difference in learning scores.

However, **statistical significance does not necessarily mean practical significance**.

A statistically significant result can have a very small difference between groups that may not matter much in the real world. Therefore, we should also examine the **actual difference in average learning scores** and determine whether that difference is large enough to be meaningful for Thanksgiving Point.

In [4]:
program_mean = program_group.mean()
control_mean = control_group.mean()
mean_difference = program_mean - control_mean

print("Program group mean:", program_mean)
print("Control group mean:", control_mean)
print("Mean difference:", mean_difference)
print("p-value:", p_value)

Program group mean: 3.8078512396694215
Control group mean: 3.1745810055865924
Mean difference: 0.6332702340828291
p-value: 1.4615568234103225e-34


This lets you tell a much stronger story:

“Guests who participated in the education program had an average learning score X points higher than guests who did not. The t-test produced a p-value of Y, indicating that this difference was/was not statistically significant.”

While the results show a strong association between program participation and higher learning scores, additional experimental evidence, such as random assignment, would be needed to determine whether the program itself caused the improvement.

Thus, we'll use _Cohen's D_

In [6]:
mean1 = program_group.mean()
mean2 = control_group.mean()

std1 = program_group.std()
std2 = control_group.std()

n1 = len(program_group)
n2 = len(control_group)

pooled_std = np.sqrt

pooled_std = np.sqrt(((n1 - 1) * std1**2  + (n2 - 1) * std2**2)
                                / (n1 + n2 - 2))

cohens_d = (mean1 - mean2) / pooled_std

print("Cohen's d:", cohens_d)


Cohen's d: 0.7447001427697089


Now I can claim "Program participants had higher average perceived learning scores than nonparticipants. The difference was statistically significant and represented a moderate effect.”

However, since the two groups aren’t necessarily comparable, maybe the program isn’t causing the entire difference.
So we control for other variables.

In [10]:
model = smf.ols("""learning_score ~ C(education_program)
                    + age
                    + family_size
                    + participated_booth
                    """,
                    data=data).fit()

print(model.summary())


                            OLS Regression Results                            
Dep. Variable:         learning_score   R-squared:                       0.119
Model:                            OLS   Adj. R-squared:                  0.116
Method:                 Least Squares   F-statistic:                     40.39
Date:                Sat, 12 Sep 2026   Prob (F-statistic):           8.91e-32
Time:                        09:44:10   Log-Likelihood:                -1506.4
No. Observations:                1200   AIC:                             3023.
Df Residuals:                    1195   BIC:                             3048.
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
                                  coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------------------
Intercept         

A multiple linear regression was used to examine whether participation in the education program was associated with higher learning scores after accounting for other factors, including age, family size, and booth participation.

The regression results showed that education program participation remained positively associated with learning scores. After accounting for the other variables in the model, guests who participated in the education program had learning scores approximately 0.63 points higher on average than guests who did not participate.

The education program coefficient was 0.6324 with a p-value < 0.001, indicating that this association was statistically significant. The 95% confidence interval for the coefficient was 0.534 to 0.731, meaning the estimated difference was consistently positive within the confidence interval.

Conclusion

After accounting for age, family size, and booth participation, education program participation remained significantly associated with higher learning scores, with program participants scoring approximately 0.63 points higher on average (p < 0.001).

However, this analysis demonstrates an association rather than causation. Although the regression accounts for several potentially relevant factors, other unmeasured factors could still influence both program participation and learning scores. Therefore, additional experimental evidence, such as randomly assigning guests to participate or not participate in the program, would be needed to make a stronger causal claim.